In [ ]:
import pandas as pd
import glob

#busca los archivos csv que empiecen por limpieza y terminen en .csv
archivos_csv = glob.glob("limpieza*.csv") 

# Combinar archivos
dataframes = []
for archivo in archivos_csv:
    df = pd.read_csv(archivo, sep=';')
    dataframes.append(df)

# Crear DataFrame unico
df_completo = pd.concat(dataframes, ignore_index=True)

# Ordenar la fecha, si existe
if 'datetime' in df_completo.columns:
    df_completo['datetime'] = pd.to_datetime(df_completo['datetime'])
    df_completo = df_completo.sort_values('datetime')

# Guardarlo en un CSV
df_completo.to_csv("Dataset_Unificado_2020_2024.csv", sep=';', index=False)
print(f"Registros(filas) totales: {len(df_completo)}")

Registros(filas) totales: 131544


In [2]:
import pandas as pd
import numpy as np
from pathlib import Path
import logging

# Configurar logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

def unificar_datasets_avanzado(ruta_demanda, ruta_produccion, ruta_precios, 
                              ruta_salida, separador=';', encoding='utf-8'):
    """
    Versión avanzada con manejo de errores y validaciones.
    """
    try:
        # Verificar que los archivos existen
        archivos = [ruta_demanda, ruta_produccion, ruta_precios]
        for archivo in archivos:
            if not Path(archivo).exists():
                raise FileNotFoundError(f"Archivo no encontrado: {archivo}")
        
        # Leer archivos con manejo de errores
        logging.info("Leyendo archivos...")
        df_demanda = pd.read_csv("LimpiezaDemanda2020_2024", sep=separador, encoding=encoding)
        df_produccion = pd.read_csv("LimpiezaGeneracionTipos2020_2024", sep=separador, encoding=encoding)
        df_precios = pd.read_csv("limpiezaPrecio3_Omie2020_2024", sep=separador, encoding=encoding)
        
        # Validar columna datetime en cada archivo
        for nombre, df in [('demanda', df_demanda), ('produccion', df_produccion), ('precios', df_precios)]:
            if 'datetime' not in df.columns:
                raise ValueError(f"El dataset de {nombre} no tiene columna 'datetime'")
        
        # Convertir datetime con manejo de errores
        logging.info("Procesando fechas...")
        for nombre, df in [('demanda', df_demanda), ('produccion', df_produccion), ('precios', df_precios)]:
            try:
                df['datetime'] = pd.to_datetime(df['datetime'])
            except Exception as e:
                logging.error(f"Error al convertir datetime en {nombre}: {e}")
                raise
        
        # Unir datasets
        logging.info("Unificando datasets...")
        
        # Primera unión: demanda + producción
        df_unificado = pd.merge(df_demanda, df_produccion, 
                               on='datetime', 
                               how='inner',
                               suffixes=('_demanda', '_produccion'))
        
        # Segunda unión: añadir precios
        df_unificado = pd.merge(df_unificado, df_precios, 
                               on='datetime', 
                               how='left',  # left join para no perder registros de demanda+producción
                               suffixes=('', '_precios'))
        
        # Limpieza de columnas duplicadas
        logging.info("Limpiando columnas duplicadas...")
        # Identificar columnas duplicadas (excluyendo datetime)
        columnas = df_unificado.columns.tolist()
        columnas_sin_datetime = [col for col in columnas if col != 'datetime']
        
        # Para cada columna, si tiene sufijo, quedarse con la primera versión
        columnas_unicas = []
        for col in columnas_sin_datetime:
            # Si la columna tiene sufijos, tomar la versión sin sufijo si existe
            base = col.split('_')[0] if '_' in col else col
            if base not in columnas_unicas:
                # Buscar la versión sin sufijo
                version_sin_sufijo = [c for c in columnas if c == base]
                if version_sin_sufijo:
                    columnas_unicas.append(base)
                else:
                    # Si no hay versión sin sufijo, tomar la primera que aparece
                    columnas_unicas.append(col)
        
        # Seleccionar solo las columnas únicas más datetime
        columnas_finales = ['datetime'] + [col for col in columnas_unicas if col in df_unificado.columns]
        df_unificado = df_unificado[columnas_finales]
        
        # Ordenar y resetear índice
        df_unificado = df_unificado.sort_values('datetime').reset_index(drop=True)
        
        # Eliminar filas con valores NaN en columnas críticas (opcional)
        # df_unificado = df_unificado.dropna(subset=['demand', 'price'])
        
        # Guardar resultado
        logging.info(f"Guardando dataset en: {ruta_salida}")
        df_unificado.to_csv(ruta_salida, sep=separador, index=False)
        
        # Reporte final
        logging.info(f"✅ Dataset unificado creado exitosamente!")
        logging.info(f"   - Registros: {len(df_unificado)}")
        logging.info(f"   - Columnas: {len(df_unificado.columns)}")
        logging.info(f"   - Rango de fechas: {df_unificado['datetime'].min()} a {df_unificado['datetime'].max()}")
        
        return df_unificado
        
    except Exception as e:
        logging.error(f"❌ Error durante el proceso: {e}")
        raise

# Ejemplo de uso
if __name__ == "__main__":
    # Configuración
    rutas = {
        'demanda': 'demanda.csv',
        'produccion': 'produccion.csv',
        'precios': 'precios.csv',
        'salida': 'dataset_unificado.csv'
    }
    
    # Ejecutar unificación
    df_final = unificar_datasets_avanzado(**rutas)
    
    # Mostrar información adicional
    print("\n📊 Estadísticas del dataset unificado:")
    print(df_final[['demand', 'price']].describe())

TypeError: unificar_datasets_avanzado() got an unexpected keyword argument 'demanda'

In [ ]:
import pandas as pd
import glob

# Busca los archivos csv que empiecen por limpieza y terminen en .csv
archivos_csv = glob.glob("limpieza*.csv")

# Verificar que se encontraron archivos
if len(archivos_csv) == 0:
    print("No se encontraron archivos que empiecen por 'limpieza'")
    print("Archivos en la carpeta actual:")
    import os
    for archivo in os.listdir('.'):
        if archivo.endswith('.csv'):
            print(f"   - {archivo}")
    exit()

print(f"Encontrados {len(archivos_csv)} archivos:")
for archivo in archivos_csv:
    print(f"   - {archivo}")

# Leer todos los archivos encontrados
dataframes = []
for archivo in archivos_csv:
    print(f"Leyendo: {archivo}")
    df = pd.read_csv(archivo, sep=';')
    print(f"   → {len(df)} registros, {len(df.columns)} columnas")
    dataframes.append(df)

# Verificar que tenemos exactamente 3 archivos (demanda, producción, precios)
if len(dataframes) != 3:
    print(f"Se esperaban 3 archivos, pero se encontraron {len(dataframes)}")
    print("   Los archivos deberían ser: limpieza_demanda.csv, limpieza_produccion.csv, limpieza_precios.csv")
    # Intentar continuar con los que se encontraron

# Unificar los datasets usando merge por datetime
print("\n Unificando datasets...")

# Empezar con el primer dataframe
df_unificado = dataframes[0]

# Unir con los siguientes dataframes uno por uno
for i in range(1, len(dataframes)):
    df_unificado = pd.merge(df_unificado, dataframes[i], 
                           on='datetime', 
                           how='inner')
    print(f"   Unión {i}: {len(df_unificado)} registros")

# Limpiar columnas duplicadas de 'hour'
columnas_hour = [col for col in df_unificado.columns if col == 'hour' or col.startswith('hour_')]
if len(columnas_hour) > 1:
    print(f"\n Eliminando columnas 'hour' duplicadas...")
    # Conservar solo la primera columna 'hour'
    if 'hour' in df_unificado.columns:
        for col in columnas_hour:
            if col != 'hour':
                df_unificado.drop(columns=[col], inplace=True)
    print(f"   Columnas eliminadas: {len(columnas_hour)-1}")

# Ordenar por fecha
if 'datetime' in df_unificado.columns:
    print("\n Ordenando por fecha...")
    df_unificado['datetime'] = pd.to_datetime(df_unificado['datetime'])
    df_unificado = df_unificado.sort_values('datetime').reset_index(drop=True)
    print(f"   Rango de fechas: {df_unificado['datetime'].min()} a {df_unificado['datetime'].max()}")

# Guardar el dataset unificado
nombre_archivo = "Dataset_Unificado_2020_2024.csv"
df_unificado.to_csv(nombre_archivo, sep=';', index=False)

# Mostrar resultados finales
print("\n" + "="*60)
print(" DATASET UNIFICADO COMPLETADO")
print("="*60)
print(f" Archivos procesados: {len(archivos_csv)}")
for archivo in archivos_csv:
    print(f"   - {archivo}")
print(f"\n Registros totales: {len(df_unificado):,}")
print(f" Columnas totales: {len(df_unificado.columns)}")
print(f" Archivo guardado: {nombre_archivo}")

print("\n Primeras 3 filas:")
print(df_unificado.head(3))

print("\n Columnas disponibles:")
for i, col in enumerate(df_unificado.columns, 1):
    print(f"   {i:2d}. {col}")

✅ Encontrados 3 archivos:
   - LimpiezaDemanda2020_2024.csv
   - LimpiezaGeneracionTipos2020_2024.csv
   - limpiezaPrecio3_Omie2020_2024.csv
📂 Leyendo: LimpiezaDemanda2020_2024.csv
   → 43848 registros, 3 columnas
📂 Leyendo: LimpiezaGeneracionTipos2020_2024.csv
   → 43848 registros, 29 columnas
📂 Leyendo: limpiezaPrecio3_Omie2020_2024.csv
   → 43848 registros, 3 columnas

🔗 Unificando datasets...
   → Unión 1: 43848 registros
   → Unión 2: 43848 registros

🧹 Eliminando columnas 'hour' duplicadas...
   → Columnas eliminadas: 2

📅 Ordenando por fecha...
   → Rango de fechas: 2020-01-01 00:00:00 a 2024-12-31 23:00:00

✅ DATASET UNIFICADO COMPLETADO
📌 Archivos procesados: 3
   - LimpiezaDemanda2020_2024.csv
   - LimpiezaGeneracionTipos2020_2024.csv
   - limpiezaPrecio3_Omie2020_2024.csv

📌 Registros totales: 43,848
📌 Columnas totales: 31
📌 Archivo guardado: Dataset_Unificado_2020_2024.csv

📋 Primeras 3 filas:
             datetime    demand  year  month  day  dayofweek  is_weekend  \
0 202